# Data Processing

### This notebook extracts different features required for synthetic data generation (includes some web scraping so run with caution)

In [1]:
import json
import pprint
import base64
import multiprocessing as mp
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

pd.set_option('display.max_columns', None)

## Explore JSON entities

In [3]:
file_path = '../Source_Data_DONOTCHANGE/entities_ftm.json'
entities_ftm = []

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        # Load each line as a dictionary
        entity = json.loads(line)
        entities_ftm.append(entity)

In [4]:
file_path = '../Source_Data_DONOTCHANGE/senzing.json'
senzing = []

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        # Load each line as a dictionary
        entity = json.loads(line)
        senzing.append(entity)

In [5]:
for json_file in entities_ftm:
    if json_file['caption'] == "ASLAN KAITMAZOV":
        print('This is the dictionary with the name match')
        print('-----------------------------------------')
        pprint.pprint(json_file)
        print('=========================================')
    if json_file['id'] =='interpol-red-bc471cd400faf98dc75741c801ccbaef07d77544':
        print('This is the dictionary with ID match')
        print('-----------------------------------------')
        pprint.pprint(json_file)
        print('=========================================')

This is the dictionary with the name match
-----------------------------------------
{'caption': 'ASLAN KAITMAZOV',
 'datasets': ['interpol_red_notices'],
 'first_seen': '2024-03-04T17:09:51',
 'id': 'NK-6UDzDx6TSr6qGFuBYBLYKt',
 'last_change': '2025-10-06T18:27:01',
 'last_seen': '2026-02-05T00:27:01',
 'origin': ['https://ws-public.interpol.int/notices/v1/red/2016-74892'],
 'properties': {'birthDate': ['1985-02-05'],
                'birthPlace': ['DAGESTANSKIE OGNI TOWN, DAGESTAN REGIION'],
                'country': ['ru'],
                'firstName': ['ASLAN'],
                'gender': ['male'],
                'lastName': ['KAITMAZOV'],
                'name': ['ASLAN KAITMAZOV'],
                'nationality': ['ru'],
                'programId': ['INTERPOL-RN'],
                'topics': ['wanted', 'crime']},
 'referents': ['interpol-red-2016-74892',
               'kg-fiu-45fd049cea9025f62d8641352fb95af63b6ea20d',
               'kzafm-49fd4d16473aa9623aa2db4a85eb7fc7ef066e7

In [6]:
entities_ftm_normalized = pd.json_normalize(entities_ftm)
entities_ftm_person = entities_ftm_normalized[entities_ftm_normalized['schema'] == 'Person'].reset_index(drop=True)
entities_ftm_person = entities_ftm_person.dropna(axis = 1, how = 'all')

In [7]:
entities_ftm_person[entities_ftm_person['id']== 'interpol-red-2011-15494']['origin']

946    [https://ws-public.interpol.int/notices/v1/red...
Name: origin, dtype: object

## WebScrape Origin Data from API

In [8]:
# functions to scrape origin API via Selenium
def make_driver(headless: bool = False):
    options = Options()
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    # If you want headless:
    if headless:
        # Chrome 109+ recommended headless mode
        options.add_argument("--headless=new")

    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options,
    )

def get_notice_data(notice_url, driver):
    """
    Directly accesses the Interpol Notice API link and returns the JSON.
    """
    try:
        driver.get(notice_url)
        # Find the JSON text rendered in the browser window
        soup = BeautifulSoup(driver.page_source, "html.parser")
        pre = soup.find("pre")
        
        if pre and pre.text:
            return json.loads(pre.text)
        return None
    except Exception as e:
        print(f"Error fetching {notice_url}: {e}")
        return None

def worker(chunk_df, worker_id):
    driver = make_driver(headless=False)
    results = []
    
    for _, row in tqdm(chunk_df.iterrows(), total=len(chunk_df), desc=f"Worker {worker_id}", position=worker_id):
        person_id, url, num_apis = row[0], row[1], row[2]
        try:
            driver.get(url)
            soup = BeautifulSoup(driver.page_source, "html.parser")
            pre = soup.find("pre")
            if pre and pre.text:
                data = json.loads(pre.text)
                df_temp = pd.json_normalize(data)
                df_temp['id'] = person_id
                df_temp['num apis'] = num_apis
                results.append(df_temp)
        except Exception as e:
            continue 
            
    driver.quit()
    return pd.concat(results) if results else pd.DataFrame()

def main(data):
    for row in origin_explode.values:
        person_id, url, num_apis = row[0], row[1], row[2]
        json_extract = get_notice_data(url, make_driver())
        df = pd.json_normalize(json_extract)
        df['id'] = person_id
        df['num apis'] = num_apis
        origin_df = pd.concat([origin_df, df])
    return origin_df

In [9]:
origin_extraction = entities_ftm_person[['id', 'origin']]
origin_extraction['num of origin apis'] = origin_extraction['origin'].apply(lambda x: len(x))
origin_explode = origin_extraction.explode('origin')
origin_explode

C:\Users\Dian\AppData\Local\Temp\ipykernel_29044\417531296.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  origin_extraction['num of origin apis'] = origin_extraction['origin'].apply(lambda x: len(x))


,id,origin,num of origin apis
0,NK-224TRezPqwzhQZ37exWxtX,https://ws-public.interpol.int/notices/v1/red/...,1
1,NK-22baaNEkbgBeZokNLmpaKd,https://ws-public.interpol.int/notices/v1/red/...,1
2,NK-282timUs47w9otBkgPN6hw,https://ws-public.interpol.int/notices/v1/red/...,1
3,NK-2Ld7pewCoLeyspMtVCMMMo,https://ws-public.interpol.int/notices/v1/red/...,1
4,NK-2SjjvqaYqkvk965QDYYMmA,https://ws-public.interpol.int/notices/v1/red/...,1
...,...,...,...
6447,interpol-red-2026-5819,https://ws-public.interpol.int/notices/v1/red/...,1
6448,interpol-red-2026-6812,https://ws-public.interpol.int/notices/v1/red/...,1
6449,interpol-red-2026-688,https://ws-public.interpol.int/notices/v1/red/...,1
6450,interpol-red-2026-6996,https://ws-public.interpol.int/notices/v1/red/...,1


In [1]:
#### DO NOT RUN THIS UNLESS YOU NEED TO REEXTRACT NETWORK DATA (Advised to run only for a few records as this will take extremely to run) ####

# extract JSON from origin API - linear loop

#origin_df = pd.read_excel('origin_part1.xlsx') # the reading was finished halfway so I saved the first half

# for row in origin_explode.values:
#     person_id, url, num_apis = row[0], row[1], row[2]
#     json_extract = get_notice_data(url, make_driver())
#     df = pd.json_normalize(json_extract)
#     df['id'] = person_id
#     df['num apis'] = num_apis
#     origin_df = pd.concat([origin_df, df])

In [11]:
origin_df

,Unnamed: 0,date_of_birth,distinguishing_marks,weight,nationalities,entity_id,eyes_colors_id,sex_id,place_of_birth,forename,arrest_warrants,country_of_birth_id,hairs_id,name,languages_spoken_ids,height,_embedded.links,_links.self.href,_links.images.href,_links.thumbnail.href,id,num apis,status,message
0,0.0,1992/06/28,NaN,0.0,['RU'],2019/108810,['OTHD'],F,KOFARNIKHON TOWN,SANAVBARI,[{'charge': 'participation in the activity of ...,TJ,['OTHD'],NIKITENKO,"['TGK', 'ARA', 'ENG', 'RUS']",0.00,"[{'forename': 'ANDREY', 'color': 'RED', 'name'...",https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,NK-224TRezPqwzhQZ37exWxtX,1,NaN,NaN
1,1.0,1989/01/05,NaN,0.0,['RU'],2016/62557,NaN,M,DAGESTAN REPUBLIC,DZHAMBULAT,[{'charge': 'Participation in illegal armed fo...,RU,NaN,GALIMOV,"['RUS', 'KUM']",0.00,[],https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,NK-22baaNEkbgBeZokNLmpaKd,1,NaN,NaN
2,2.0,1982/05/24,NaN,0.0,['RU'],2021/16004,['BLA'],F,"GROZNY, CHECHEN REPUBLIC",KHAVA,[{'charge': 'Participation in illegal armed fo...,RU,['BLA'],EFENDIEVA,['RUS'],1.75,[],https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,NK-282timUs47w9otBkgPN6hw,1,NaN,NaN
3,3.0,1977/07/13,NaN,0.0,['CN'],2010/24537,NaN,M,LIAONING,JIAN,[{'charge': '1) Murder\r\n2) Manslaughter\r\n3...,CN,NaN,XIA,"['CHI', 'ENG']",0.00,[],https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,NK-2Ld7pewCoLeyspMtVCMMMo,1,NaN,NaN
4,4.0,1978/12/28,NaN,NaN,['PL'],2025/55066,NaN,M,NaN,NORBERT,"[{'charge': 'assaulting a public officer, impe...",PL,NaN,BIALAS,NaN,1.80,[],https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,NK-2SjjvqaYqkvk965QDYYMmA,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,NaN,1978/01/04,None,NaN,[CI],2026/5819,None,M,TREICHVILLE,AHMED,"[{'charge': 'MEURTRE', 'issuing_country_id': '...",CI,None,SOUMAHORO,[FRE],NaN,[],https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,interpol-red-2026-5819,1,NaN,NaN
0,NaN,1978/10/22,None,NaN,[PL],2026/6812,None,M,WARSZAWA,MICHAL ZBIGNIEW,"[{'charge': 'Tax fraud Invoice forgery', 'iss...",PL,None,BARYS,None,NaN,[],https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,interpol-red-2026-6812,1,NaN,NaN
0,NaN,1979/01/19,None,NaN,[NO],2026/688,[BRO],M,Gujranwala,MOHAMMAD SHAHZAD,[{'charge': 'Norwegian Penal Code (1902) secti...,PK,[BLA],GULL,None,1.80,[],https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,interpol-red-2026-688,1,NaN,NaN
0,NaN,1986/05/20,None,NaN,[ME],2026/6996,None,M,KOLASIN,MILOS,[{'charge': '– Establishment of a Criminal Org...,ME,None,MEDENICA,[SRP],NaN,[],https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,interpol-red-2026-6996,1,NaN,NaN


In [12]:
origin_df = origin_df.reset_index(drop = True).drop(columns = ['Unnamed: 0'])

In [17]:
origin_df.to_excel('processed/origin_complete.xlsx', index=False)

## Read the resulting dataframe (After Webscraping)

In [2]:
origin_df = pd.read_excel('processed/origin_complete.xlsx')
origin_df

,date_of_birth,distinguishing_marks,weight,nationalities,entity_id,eyes_colors_id,sex_id,place_of_birth,forename,arrest_warrants,country_of_birth_id,hairs_id,name,languages_spoken_ids,height,_embedded.links,_links.self.href,_links.images.href,_links.thumbnail.href,id,num apis,status,message
0,1992/06/28,NaN,0.0,['RU'],2019/108810,['OTHD'],F,KOFARNIKHON TOWN,SANAVBARI,[{'charge': 'participation in the activity of ...,TJ,['OTHD'],NIKITENKO,"['TGK', 'ARA', 'ENG', 'RUS']",0.00,"[{'forename': 'ANDREY', 'color': 'RED', 'name'...",https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,NK-224TRezPqwzhQZ37exWxtX,1,NaN,NaN
1,1989/01/05,NaN,0.0,['RU'],2016/62557,NaN,M,DAGESTAN REPUBLIC,DZHAMBULAT,[{'charge': 'Participation in illegal armed fo...,RU,NaN,GALIMOV,"['RUS', 'KUM']",0.00,[],https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,NK-22baaNEkbgBeZokNLmpaKd,1,NaN,NaN
2,1982/05/24,NaN,0.0,['RU'],2021/16004,['BLA'],F,"GROZNY, CHECHEN REPUBLIC",KHAVA,[{'charge': 'Participation in illegal armed fo...,RU,['BLA'],EFENDIEVA,['RUS'],1.75,[],https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,NK-282timUs47w9otBkgPN6hw,1,NaN,NaN
3,1977/07/13,NaN,0.0,['CN'],2010/24537,NaN,M,LIAONING,JIAN,[{'charge': '1) Murder\r\n2) Manslaughter\r\n3...,CN,NaN,XIA,"['CHI', 'ENG']",0.00,[],https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,NK-2Ld7pewCoLeyspMtVCMMMo,1,NaN,NaN
4,1978/12/28,NaN,NaN,['PL'],2025/55066,NaN,M,NaN,NORBERT,"[{'charge': 'assaulting a public officer, impe...",PL,NaN,BIALAS,NaN,1.80,[],https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,NK-2SjjvqaYqkvk965QDYYMmA,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6452,1978/01/04,NaN,NaN,['CI'],2026/5819,NaN,M,TREICHVILLE,AHMED,"[{'charge': 'MEURTRE', 'issuing_country_id': '...",CI,NaN,SOUMAHORO,['FRE'],NaN,[],https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,interpol-red-2026-5819,1,NaN,NaN
6453,1978/10/22,NaN,NaN,['PL'],2026/6812,NaN,M,WARSZAWA,MICHAL ZBIGNIEW,"[{'charge': 'Tax fraud\r\nInvoice forgery', 'i...",PL,NaN,BARYS,NaN,NaN,[],https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,interpol-red-2026-6812,1,NaN,NaN
6454,1979/01/19,NaN,NaN,['NO'],2026/688,['BRO'],M,Gujranwala,MOHAMMAD SHAHZAD,[{'charge': 'Norwegian Penal Code (1902) secti...,PK,['BLA'],GULL,NaN,1.80,[],https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,interpol-red-2026-688,1,NaN,NaN
6455,1986/05/20,NaN,NaN,['ME'],2026/6996,NaN,M,KOLASIN,MILOS,[{'charge': '– Establishment of a Criminal Org...,ME,NaN,MEDENICA,['SRP'],NaN,[],https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,https://ws-public.interpol.int/notices/v1/red/...,interpol-red-2026-6996,1,NaN,NaN


In [8]:
origin_df[~origin_df['status'].isna()].count()

date_of_birth             0
distinguishing_marks      0
weight                    0
nationalities             0
entity_id                 0
eyes_colors_id            0
sex_id                    0
place_of_birth            0
forename                  0
arrest_warrants           0
country_of_birth_id       0
hairs_id                  0
name                     13
languages_spoken_ids      0
height                    0
_embedded.links           0
_links.self.href          0
_links.images.href        0
_links.thumbnail.href     0
id                       13
num apis                 13
status                   13
message                  13
dtype: int64

In [6]:
origin_df[~origin_df['status'].isna()]['message'].unique()

array(['Unable to process your query. Error message have been logged.'],
      dtype=object)

In [ ]:
origin_df